In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r'C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\business_crisis_cohort.csv')

In [3]:
df.sort_values(by=['t0'], inplace=True)

In [4]:
df

,관리번호,구,읍면동,업종,기준분기,t0,업력_일수,매출액,이용건수,sales_yoy,...,average_ticket,horizon_end_12M,Y_12M,label_status_12M,horizon_end_18M,Y_18M,label_status_18M,horizon_end_24M,Y_24M,label_status_24M
48562,3460000-104-2010-00068,수성구,수성4가동,식품_휴게음식점,2016Q1,2016-03-31,2095,142855816.0,17613,-0.294233,...,8110.816783,2017-03-31,1.0,observed,2017-09-30,1.0,observed,2018-03-31,1.0,observed
41667,345000000920050124,북구,무태조야동,식품_축산판매업,2016Q1,2016-03-31,3755,704588446.0,15927,-0.229347,...,44238.616563,2017-03-31,0.0,observed,2017-09-30,0.0,observed,2018-03-31,0.0,observed
41663,345000000920050115,북구,산격2동,식품_축산판매업,2016Q1,2016-03-31,3780,563591132.0,12183,-0.343828,...,46260.455717,2017-03-31,1.0,observed,2017-09-30,1.0,observed,2018-03-31,1.0,observed
59257,3470000-101-2016-00042,달서구,이곡2동,식품_일반음식점,2016Q1,2016-03-31,55,518471594.0,21515,-0.533781,...,24098.145201,2017-03-31,1.0,observed,2017-09-30,1.0,observed,2018-03-31,1.0,observed
48847,3460000-104-2014-00190,수성구,수성4가동,식품_휴게음식점,2016Q1,2016-03-31,486,142855816.0,17613,-0.294233,...,8110.816783,2017-03-31,1.0,observed,2017-09-30,1.0,observed,2018-03-31,1.0,observed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72904,3480000-101-2022-00275,달성군,다사읍,식품_일반음식점,2025Q4,2025-12-31,1118,600237283.0,20546,-0.442391,...,29214.313394,2026-12-31,NaN,insufficient_followup,2027-06-30,NaN,insufficient_followup,2027-12-31,NaN,insufficient_followup
72905,3480000-101-2022-00276,달성군,다사읍,식품_일반음식점,2025Q4,2025-12-31,1115,600237283.0,20546,-0.442391,...,29214.313394,2026-12-31,NaN,insufficient_followup,2027-06-30,NaN,insufficient_followup,2027-12-31,NaN,insufficient_followup
72906,3480000-101-2022-00277,달성군,다사읍,식품_일반음식점,2025Q4,2025-12-31,1111,600237283.0,20546,-0.442391,...,29214.313394,2026-12-31,NaN,insufficient_followup,2027-06-30,NaN,insufficient_followup,2027-12-31,NaN,insufficient_followup
72843,3480000-101-2022-00177,달성군,다사읍,식품_일반음식점,2025Q4,2025-12-31,1223,600237283.0,20546,-0.442391,...,29214.313394,2026-12-31,NaN,insufficient_followup,2027-06-30,NaN,insufficient_followup,2027-12-31,NaN,insufficient_followup


In [5]:
from pathlib import Path
import pandas as pd
import re


# ============================================================
# 0. 설정
# ============================================================

BASE_DIR = Path(
    r"C:\Users\DC\2026\IMBANK\iM_Blockchain_AI"
    r"\data\processed\variable\서비스인구"
)

START_YEAR = 2017
END_YEAR = 2025


# ============================================================
# 1. 문자열 정리 함수
# ============================================================

def clean_text(x):
    """
    읍면동 등의 문자열에서 앞뒤/중간 공백 제거
    """
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()
    x = re.sub(r"\s+", "", x)

    return x


# ============================================================
# 2. 시트명 자동 탐색
# ============================================================

def find_sheet_name(sheet_names, keywords):
    """
    keywords 중 하나가 포함된 시트를 찾음.

    예:
    ["시간"] → 시간대별, 시간대별 분석, 시간대 분석
    ["성연령", "연령"] → 성연령별, 연령대별 분석 등
    """

    for sheet in sheet_names:

        clean_sheet = re.sub(r"\s+", "", str(sheet))

        if any(keyword in clean_sheet for keyword in keywords):
            return sheet

    return None


# ============================================================
# 3. 컬럼명 자동 탐색
# ============================================================

def find_column(columns, candidates):
    """
    파일마다 컬럼명이 약간 다른 경우 대응.

    예:
    분석영역 / 읍면동 / 행정동
    기준년월 / 기준연월 / 년월
    """

    # 비교용 정규화
    normalized = {
        re.sub(r"\s+", "", str(col)): col
        for col in columns
    }

    # 1차: 정확히 일치
    for candidate in candidates:

        candidate_clean = re.sub(r"\s+", "", candidate)

        if candidate_clean in normalized:
            return normalized[candidate_clean]

    # 2차: 일부 문자열 포함
    for col in columns:

        clean_col = re.sub(r"\s+", "", str(col))

        for candidate in candidates:

            candidate_clean = re.sub(r"\s+", "", candidate)

            if candidate_clean in clean_col:
                return col

    return None


# ============================================================
# 4. 하나의 Excel 파일 읽는 함수
# ============================================================

def read_service_population(file):

    try:

        # ----------------------------------------------------
        # Excel 구조 확인
        # ----------------------------------------------------

        xls = pd.ExcelFile(file)

        sheet_names = xls.sheet_names

        # ----------------------------------------------------
        # 시간대 시트 탐색
        # ----------------------------------------------------

        time_sheet = find_sheet_name(
            sheet_names,
            [
                "시간대별",
                "시간대",
                "시간"
            ]
        )

        # ----------------------------------------------------
        # 성연령 시트 탐색
        # ----------------------------------------------------

        age_sheet = find_sheet_name(
            sheet_names,
            [
                "성연령별",
                "성연령",
                "연령대별",
                "연령"
            ]
        )

        print(
            f"{file.name}"
            f" | 시간={time_sheet}"
            f" | 연령={age_sheet}"
        )

        # ----------------------------------------------------
        # 필요한 시트가 없으면 건너뜀
        # ----------------------------------------------------

        if time_sheet is None:
            print("   [경고] 시간대 관련 시트 없음")
            print("   실제 시트:", sheet_names)
            return None

        if age_sheet is None:
            print("   [경고] 연령 관련 시트 없음")
            print("   실제 시트:", sheet_names)
            return None

        # ----------------------------------------------------
        # 시트 읽기
        # ----------------------------------------------------

        time_df = pd.read_excel(
            file,
            sheet_name=time_sheet
        )

        age_df = pd.read_excel(
            file,
            sheet_name=age_sheet
        )

        # ----------------------------------------------------
        # 컬럼명 탐색
        # ----------------------------------------------------

        time_date_col = find_column(
            time_df.columns,
            [
                "기준년월",
                "기준연월",
                "년월"
            ]
        )

        time_area_col = find_column(
            time_df.columns,
            [
                "분석영역",
                "읍면동",
                "행정동"
            ]
        )

        age_date_col = find_column(
            age_df.columns,
            [
                "기준년월",
                "기준연월",
                "년월"
            ]
        )

        age_area_col = find_column(
            age_df.columns,
            [
                "분석영역",
                "읍면동",
                "행정동"
            ]
        )

        # ----------------------------------------------------
        # 필요한 컬럼 존재 확인
        # ----------------------------------------------------

        if time_date_col is None:
            raise ValueError(
                f"시간대 시트에서 기준년월 컬럼을 찾을 수 없음: "
                f"{list(time_df.columns)}"
            )

        if time_area_col is None:
            raise ValueError(
                f"시간대 시트에서 읍면동 컬럼을 찾을 수 없음: "
                f"{list(time_df.columns)}"
            )

        if age_date_col is None:
            raise ValueError(
                f"성연령 시트에서 기준년월 컬럼을 찾을 수 없음: "
                f"{list(age_df.columns)}"
            )

        if age_area_col is None:
            raise ValueError(
                f"성연령 시트에서 읍면동 컬럼을 찾을 수 없음: "
                f"{list(age_df.columns)}"
            )

        # ----------------------------------------------------
        # 컬럼명 표준화
        # ----------------------------------------------------

        time_df = time_df.rename(
            columns={
                time_date_col: "기준년월",
                time_area_col: "읍면동"
            }
        )

        age_df = age_df.rename(
            columns={
                age_date_col: "기준년월",
                age_area_col: "읍면동"
            }
        )

        # ----------------------------------------------------
        # 읍면동 정리
        # ----------------------------------------------------

        time_df["읍면동"] = (
            time_df["읍면동"]
            .apply(clean_text)
        )

        age_df["읍면동"] = (
            age_df["읍면동"]
            .apply(clean_text)
        )

        # ----------------------------------------------------
        # 기준년월 정리
        #
        # 202506
        # "202506"
        # 2025-06
        # 등의 경우를 최대한 대응
        # ----------------------------------------------------

        def normalize_yyyymm(series):

            s = series.astype(str)

            # 숫자만 남김
            s = s.str.replace(
                r"[^0-9]",
                "",
                regex=True
            )

            # 앞 6자리
            s = s.str[:6]

            return pd.to_numeric(
                s,
                errors="coerce"
            ).astype("Int64")

        time_df["기준년월"] = normalize_yyyymm(
            time_df["기준년월"]
        )

        age_df["기준년월"] = normalize_yyyymm(
            age_df["기준년월"]
        )

        # ----------------------------------------------------
        # 잘못된 행 제거
        # ----------------------------------------------------

        time_df = time_df.dropna(
            subset=[
                "기준년월",
                "읍면동"
            ]
        )

        age_df = age_df.dropna(
            subset=[
                "기준년월",
                "읍면동"
            ]
        )

        # ----------------------------------------------------
        # 혹시 같은 키가 중복되어 있는지 확인
        # ----------------------------------------------------

        time_dup = time_df.duplicated(
            subset=[
                "기준년월",
                "읍면동"
            ],
            keep=False
        )

        if time_dup.any():

            print(
                f"   [주의] 시간대 중복 "
                f"{time_dup.sum()}행"
            )

            time_df = time_df.drop_duplicates(
                subset=[
                    "기준년월",
                    "읍면동"
                ],
                keep="first"
            )

        age_dup = age_df.duplicated(
            subset=[
                "기준년월",
                "읍면동"
            ],
            keep=False
        )

        if age_dup.any():

            print(
                f"   [주의] 성연령 중복 "
                f"{age_dup.sum()}행"
            )

            age_df = age_df.drop_duplicates(
                subset=[
                    "기준년월",
                    "읍면동"
                ],
                keep="first"
            )

        # ----------------------------------------------------
        # 시간대 + 성연령 결합
        # ----------------------------------------------------

        monthly = time_df.merge(
            age_df,
            on=[
                "기준년월",
                "읍면동"
            ],
            how="outer",
            validate="one_to_one"
        )

        return monthly

    except Exception as e:

        print(f"\n[오류] {file}")
        print(f"   → {e}")

        return None


# ============================================================
# 5. 2017 ~ 2025 전체 파일 읽기
# ============================================================

service_list = []

success_files = []
failed_files = []

for year in range(
    START_YEAR,
    END_YEAR + 1
):

    year_dir = (
        BASE_DIR
        / f"{year}"
    )

    print()
    print("=" * 70)
    print(year)
    print("=" * 70)

    if not year_dir.exists():

        print(
            f"[폴더 없음] "
            f"{year_dir}"
        )

        continue

    # --------------------------------------------------------
    # 해당 연도 xlsx 전체 검색
    # --------------------------------------------------------

    files = sorted(
        year_dir.glob("*.xlsx")
    )

    print(
        f"{len(files)}개 파일 발견"
    )

    # --------------------------------------------------------
    # 파일 하나씩 처리
    # --------------------------------------------------------

    for file in files:

        monthly = read_service_population(
            file
        )

        if monthly is not None:

            service_list.append(
                monthly
            )

            success_files.append(
                str(file)
            )

        else:

            failed_files.append(
                str(file)
            )


# ============================================================
# 6. 읽은 파일이 하나도 없으면 중단
# ============================================================

if len(service_list) == 0:

    raise RuntimeError(
        "읽기에 성공한 서비스인구 파일이 없습니다."
    )


# ============================================================
# 7. 모든 연월 합치기
# ============================================================

service_df = pd.concat(
    service_list,
    ignore_index=True
)

print()
print("=" * 70)
print("서비스인구 통합 결과")
print("=" * 70)

print(
    "성공 파일:",
    len(success_files)
)

print(
    "실패 파일:",
    len(failed_files)
)

print(
    "서비스인구 데이터 크기:",
    service_df.shape
)


# ============================================================
# 8. 전체 데이터 중복 확인
# ============================================================

duplicate_mask = service_df.duplicated(
    subset=[
        "기준년월",
        "읍면동"
    ],
    keep=False
)

if duplicate_mask.any():

    print()
    print(
        "[주의] 전체 데이터에서 "
        "기준년월 + 읍면동 중복 발견"
    )

    print(
        service_df.loc[
            duplicate_mask,
            [
                "기준년월",
                "읍면동"
            ]
        ]
        .sort_values(
            [
                "기준년월",
                "읍면동"
            ]
        )
        .head(30)
    )

    # 완전히 동일한 월/동 키가 여러 개라면
    # 우선 첫 번째 사용
    service_df = (
        service_df
        .drop_duplicates(
            subset=[
                "기준년월",
                "읍면동"
            ],
            keep="first"
        )
    )


# ============================================================
# 9. 서비스인구 변수에 prefix 붙이기
#
# 예:
# 00시 → 서비스인구_00시
# 10대 남성(명) → 서비스인구_10대 남성(명)
# ============================================================

key_columns = {
    "기준년월",
    "읍면동"
}

rename_dict = {
    col: f"서비스인구_{col}"
    for col in service_df.columns
    if col not in key_columns
}

service_df = service_df.rename(
    columns=rename_dict
)


# ============================================================
# 10. 기존 df 준비
# ============================================================

# t0 날짜형 변환
df["t0"] = pd.to_datetime(
    df["t0"],
    errors="coerce"
)

# 읍면동 정리
df["읍면동"] = (
    df["읍면동"]
    .apply(clean_text)
)

# ------------------------------------------------------------
# t0 → YYYYMM
#
# 2017-01-01 → 201701
# 2025-06-01 → 202506
# ------------------------------------------------------------

df["_기준년월"] = (
    df["t0"].dt.year * 100
    +
    df["t0"].dt.month
).astype("Int64")

# ============================================================
# 서비스인구 월 누적 → 일평균 + 구성비율
# ============================================================

# 기준년월 → 해당 월의 일수 계산
service_df["_date"] = pd.to_datetime(
    service_df["기준년월"].astype(str),
    format="%Y%m"
)

service_df["_days_in_month"] = (
    service_df["_date"].dt.days_in_month
)


# ============================================================
# 1. 시간대 변수 찾기
# ============================================================

# 00시 ~ 23시
time_cols = [
    col for col in service_df.columns
    if re.fullmatch(r"\d{2}시", str(col))
]

print("시간대 변수:")
print(time_cols)


# ============================================================
# 2. 성연령 변수 찾기
# ============================================================

# 예:
# 10대 남성(명)
# 20대 여성(명)
# ...
age_cols = [
    col for col in service_df.columns
    if (
        ("남성" in str(col) or "여성" in str(col))
        and "대" in str(col)
    )
]

print("성연령 변수:")
print(age_cols)


# ============================================================
# 3. 숫자형으로 변환
# ============================================================

for col in time_cols + age_cols:
    service_df[col] = pd.to_numeric(
        service_df[col],
        errors="coerce"
    )


# ============================================================
# 4. 시간대 일평균
# ============================================================

for col in time_cols:

    service_df[f"{col}_일평균"] = (
        service_df[col]
        / service_df["_days_in_month"]
    )


# ============================================================
# 5. 시간대 구성비율
# ============================================================

service_df["_시간대전체"] = (
    service_df[time_cols]
    .sum(axis=1, min_count=1)
)

for col in time_cols:

    service_df[f"{col}_비율"] = (
        service_df[col]
        / service_df["_시간대전체"]
    )


# ============================================================
# 6. 성연령 일평균
# ============================================================

for col in age_cols:

    service_df[f"{col}_일평균"] = (
        service_df[col]
        / service_df["_days_in_month"]
    )


# ============================================================
# 7. 성연령 구성비율
# ============================================================

service_df["_성연령전체"] = (
    service_df[age_cols]
    .sum(axis=1, min_count=1)
)

for col in age_cols:

    service_df[f"{col}_비율"] = (
        service_df[col]
        / service_df["_성연령전체"]
    )


# ============================================================
# 8. 원본 누적값 제거
# ============================================================

service_df = service_df.drop(
    columns=time_cols + age_cols
)


# 계산용 임시 컬럼 제거
service_df = service_df.drop(
    columns=[
        "_date",
        "_days_in_month",
        "_시간대전체",
        "_성연령전체"
    ]
)
# ============================================================
# 11. 서비스인구 병합
# ============================================================

df = df.merge(
    service_df,
    left_on=[
        "_기준년월",
        "읍면동"
    ],
    right_on=[
        "기준년월",
        "읍면동"
    ],
    how="left",
    validate="many_to_one"
)


# ============================================================
# 12. 임시 컬럼 제거
# ============================================================

df = df.drop(
    columns=[
        "_기준년월",
        "기준년월"
    ]
)


# ============================================================
# 13. 결과 확인
# ============================================================

service_cols = [
    col
    for col in df.columns
    if col.startswith(
        "서비스인구_"
    )
]

print()
print("=" * 70)
print("최종 결과")
print("=" * 70)

print(
    "최종 df:",
    df.shape
)

print(
    "추가된 서비스인구 변수:",
    len(service_cols)
)

print()
print("추가 변수:")
print(service_cols)


# ============================================================
# 14. 연도별 매칭률 확인
# ============================================================

df["_연도"] = (
    df["t0"]
    .dt.year
)

if len(service_cols) > 0:

    match_check = (
        df.groupby("_연도")
        [service_cols[0]]
        .agg(
            전체="size",
            매칭="count"
        )
    )

    match_check["매칭률"] = (
        match_check["매칭"]
        /
        match_check["전체"]
        * 100
    ).round(2)

    print()
    print("=" * 70)
    print("연도별 서비스인구 매칭률")
    print("=" * 70)

    print(
        match_check
    )

df = df.drop(
    columns="_연도"
)


# ============================================================
# 15. 2016년 확인
# ============================================================

if len(service_cols) > 0:

    check_2016 = df[
        df["t0"].dt.year == 2016
    ]

    print()
    print("=" * 70)
    print("2016년 확인")
    print("=" * 70)

    print(
        check_2016[
            [
                "t0",
                "읍면동"
            ]
            +
            service_cols[:3]
        ].head(10)
    )


# ============================================================
# 16. 샘플 확인
# ============================================================

print()
print("=" * 70)
print("최종 데이터 샘플")
print("=" * 70)

print(
    df[
        [
            "t0",
            "읍면동"
        ]
        +
        service_cols[:10]
    ].head(20)
)



2017
12개 파일 발견
서비스인구 통계_201701.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201702.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201703.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201704.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201705.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201706.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201707.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201708.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201709.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201710.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201711.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201712.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석

2018
12개 파일 발견
서비스인구 통계_201801.xlsx | 시간=시간대별 분석 | 연령=성연령별 분석
서비스인구 통계_201802.xlsx | 시간=시간대별 분석(수정) | 연령=성연령별 분석
서비스인구 통계_201803.xlsx | 시간=시간대별 분석(수정) | 연령=성연령별 분석
서비스인구 통계_201804.xlsx | 시간=시간대별 분석(수정) | 연령=성연령별 분석
서비스인구 통계_201805.xlsx | 시간=시간대별 분석(수정) | 연령=성연령별 분석
서비스인구 통계_201806.xlsx | 시간=시간대별 분석(수정) | 연령=성연령별 분석
서비스인구 통계_201807.xlsx | 시간=시간대별 분석(수정) | 연령=성연령별 분석
서비스인구 통계_201808.xlsx | 시간=시간대별 분석(수정) | 연령=성연령별 분석


In [6]:
df.to_csv(r'C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\business_crisis_cohort_v2.csv', index=False)

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


# ============================================================
# 1. 설정
# ============================================================

POP_PATH = Path(
    r"C:\Users\DC\2026\IMBANK\iM_Blockchain_AI"
    r"\data\processed\variable\등록인구\203_DT_B40003_20260918152856.csv"
)

# 기존 기준 데이터프레임은 df라고 가정
# df.columns 안에 반드시:
# t0, 읍면동
# 이 있어야 함


# ============================================================
# 2. 인구 CSV 읽기
# ============================================================

# 공공데이터 CSV는 cp949인 경우가 많아서 우선 cp949
try:
    pop_raw = pd.read_csv(
        POP_PATH,
        encoding="cp949"
    )
except UnicodeDecodeError:
    pop_raw = pd.read_csv(
        POP_PATH,
        encoding="utf-8-sig"
    )


print("원본 크기:", pop_raw.shape)
print("\n컬럼:")
print(pop_raw.columns.tolist())

print("\n인구현황별 값:")
print(
    pop_raw["인구현황별"]
    .dropna()
    .unique()
)


# ============================================================
# 3. 컬럼명 공백 정리
# ============================================================

pop_raw.columns = (
    pop_raw.columns
    .astype(str)
    .str.strip()
)

pop_raw["동·읍·면별"] = (
    pop_raw["동·읍·면별"]
    .astype(str)
    .str.strip()
)

pop_raw["인구현황별"] = (
    pop_raw["인구현황별"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 4. 사용할 변수 선택
# ============================================================
#
# 남 / 여는 원본에서
#
# 등록인구
#   ├ 남
#   └ 여
#
# 한국인
#   ├ 남
#   └ 여
#
# 외국인
#   ├ 남
#   └ 여
#
# 형태로 반복되므로 일단 제외
#
# 중복 없이 바로 사용할 수 있는 변수만 선택
# ============================================================

USE_VARIABLES = [
    "세대수",
    "등록인구",
    "한국인",
    "외국인",
    "세대당 인구",
    "65세이상고령자",
    "평균연령",
    "인구밀도",
    "면적"
]

pop = pop_raw[
    pop_raw["인구현황별"].isin(USE_VARIABLES)
].copy()


# ============================================================
# 5. 구 단위 행 제거
# ============================================================
#
# 이미지에서:
#
# 중구
# 동인동
# 삼덕동
# ...
#
# 구조이므로 '중구' 같은 구 전체 집계 행 제거
#
# 기존 df에 실제 존재하는 읍면동만 남기는 방식이
# 가장 안전함
# ============================================================

valid_dongs = set(
    df["읍면동"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

pop = pop[
    pop["동·읍·면별"].isin(valid_dongs)
].copy()


print(
    "\n매칭되는 읍면동 수:",
    pop["동·읍·면별"].nunique()
)


# ============================================================
# 6. 연도 컬럼 찾기
# ============================================================
#
# 2016 년
# 2017 년
# ...
# 형태 자동 탐색
# ============================================================

year_cols = [
    col
    for col in pop.columns
    if re.fullmatch(
        r"\d{4}\s*년",
        str(col).strip()
    )
]

print("\n발견된 연도:")
print(year_cols)


# ============================================================
# 7. Wide → Long
# ============================================================
#
# 기존:
#
# 동인동 | 등록인구 | 2016년 | 2017년 | ...
#
# ↓
#
# 동인동 | 등록인구 | 2016 | 10687
# 동인동 | 등록인구 | 2017 | 10437
#
# ============================================================

pop_long = pop.melt(
    id_vars=[
        "동·읍·면별",
        "인구현황별"
    ],
    value_vars=year_cols,
    var_name="연도",
    value_name="값"
)


# ============================================================
# 8. 연도 숫자로 변환
# ============================================================

pop_long["연도"] = (
    pop_long["연도"]
    .astype(str)
    .str.extract(r"(\d{4})")[0]
)

pop_long["연도"] = pd.to_numeric(
    pop_long["연도"],
    errors="coerce"
).astype("Int64")


# ============================================================
# 9. 값 숫자형 변환
# ============================================================
#
# 혹시 ',' 등이 들어가 있는 경우 대응
# ============================================================

pop_long["값"] = (
    pop_long["값"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)

pop_long["값"] = pd.to_numeric(
    pop_long["값"],
    errors="coerce"
)


# ============================================================
# 10. 읍면동 컬럼명 통일
# ============================================================

pop_long = pop_long.rename(
    columns={
        "동·읍·면별": "읍면동"
    }
)


# ============================================================
# 11. Long → 분석용 Wide
# ============================================================
#
# 결과:
#
# 읍면동 | 연도 | 등록인구 | 세대수 | 평균연령 | ...
#
# ============================================================

pop_year = (
    pop_long
    .pivot_table(
        index=[
            "읍면동",
            "연도"
        ],
        columns="인구현황별",
        values="값",
        aggfunc="first"
    )
    .reset_index()
)

pop_year.columns.name = None


# ============================================================
# 12. 변수명 정리
# ============================================================

rename_dict = {
    "등록인구": "인구_등록인구",
    "세대수": "인구_세대수",
    "한국인": "인구_한국인",
    "외국인": "인구_외국인",
    "세대당 인구": "인구_세대당인구",
    "65세이상고령자": "인구_65세이상",
    "평균연령": "인구_평균연령",
    "인구밀도": "인구_인구밀도",
    "면적": "인구_면적"
}

pop_year = pop_year.rename(
    columns=rename_dict
)


# ============================================================
# 13. 파생변수 생성
# ============================================================

# 고령인구 비율
if {
    "인구_65세이상",
    "인구_등록인구"
}.issubset(pop_year.columns):

    pop_year["인구_고령인구비율"] = (
        pop_year["인구_65세이상"]
        / pop_year["인구_등록인구"]
    )


# 외국인 비율
if {
    "인구_외국인",
    "인구_등록인구"
}.issubset(pop_year.columns):

    pop_year["인구_외국인비율"] = (
        pop_year["인구_외국인"]
        / pop_year["인구_등록인구"]
    )


# 한국인 비율
if {
    "인구_한국인",
    "인구_등록인구"
}.issubset(pop_year.columns):

    pop_year["인구_한국인비율"] = (
        pop_year["인구_한국인"]
        / pop_year["인구_등록인구"]
    )


# ============================================================
# 14. inf 처리
# ============================================================

pop_year = pop_year.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 15. 중복 키 검사
# ============================================================

dup = pop_year.duplicated(
    subset=[
        "읍면동",
        "연도"
    ],
    keep=False
)

if dup.any():

    print("\n[주의] 읍면동 + 연도 중복 발견")

    print(
        pop_year.loc[
            dup,
            [
                "읍면동",
                "연도"
            ]
        ]
        .sort_values(
            [
                "읍면동",
                "연도"
            ]
        )
    )

else:

    print(
        "\n읍면동 + 연도 중복 없음"
    )


# ============================================================
# 16. 기준 df 준비
# ============================================================

df["t0"] = pd.to_datetime(
    df["t0"],
    errors="coerce"
)

df["읍면동"] = (
    df["읍면동"]
    .astype(str)
    .str.strip()
)

df["_연도"] = (
    df["t0"]
    .dt.year
    .astype("Int64")
)


# ============================================================
# 17. 기존 df에 인구통계 LEFT JOIN
# ============================================================
#
# df는 절대 행을 삭제하지 않음.
#
# 예:
# 2019-01 동인동
# 2019-02 동인동
# ...
# 모두 2019년 동인동 인구통계가 붙음
#
# ============================================================

before_rows = len(df)

df = df.merge(
    pop_year,
    left_on=[
        "읍면동",
        "_연도"
    ],
    right_on=[
        "읍면동",
        "연도"
    ],
    how="left",
    validate="many_to_one"
)

after_rows = len(df)


# ============================================================
# 18. 임시 컬럼 제거
# ============================================================

df = df.drop(
    columns=[
        "_연도",
        "연도"
    ]
)


# ============================================================
# 19. 결과 확인
# ============================================================

print()
print("=" * 70)
print("병합 결과")
print("=" * 70)

print(
    f"병합 전 행 수: {before_rows:,}"
)

print(
    f"병합 후 행 수: {after_rows:,}"
)

if before_rows == after_rows:
    print("✓ 기존 df 행 수 유지")
else:
    print("⚠ 행 수가 변경됨")


# ============================================================
# 20. 추가된 인구 변수 확인
# ============================================================

population_cols = [
    col
    for col in df.columns
    if col.startswith("인구_")
]

print()
print("추가된 변수:")
print(population_cols)


# ============================================================
# 21. 연도별 매칭률 확인
# ============================================================

df["_check_year"] = df["t0"].dt.year

match_check = (
    df.groupby("_check_year")
    ["인구_등록인구"]
    .agg(
        전체="size",
        매칭="count"
    )
)

match_check["매칭률(%)"] = (
    match_check["매칭"]
    / match_check["전체"]
    * 100
).round(2)

print()
print("=" * 70)
print("연도별 인구통계 매칭률")
print("=" * 70)

print(match_check)

df = df.drop(
    columns="_check_year"
)


# ============================================================
# 22. 최종 샘플
# ============================================================

print()
print(
    df[
        [
            "t0",
            "읍면동",
            "인구_등록인구",
            "인구_세대수",
            "인구_세대당인구",
            "인구_65세이상",
            "인구_고령인구비율",
            "인구_평균연령",
            "인구_인구밀도",
            "인구_외국인비율"
        ]
    ].head(20)
)

원본 크기: (2433, 14)

컬럼:
['동·읍·면별', '인구현황별', '항목', '단위', '2016 년', '2017 년', '2018 년', '2019 년', '2020 년', '2021 년', '2022 년', '2023 년', '2024 년', 'Unnamed: 13']

인구현황별 값:
['세대수' '등록인구' '남' '여' '한국인' '외국인' '세대당 인구' '65세이상고령자' '평균연령' '인구밀도' '면적']

매칭되는 읍면동 수: 144

발견된 연도:
['2016 년', '2017 년', '2018 년', '2019 년', '2020 년', '2021 년', '2022 년', '2023 년', '2024 년']

읍면동 + 연도 중복 없음

병합 결과
병합 전 행 수: 77,736
병합 후 행 수: 77,736
✓ 기존 df 행 수 유지

추가된 변수:
['인구_65세이상', '인구_등록인구', '인구_면적', '인구_세대당인구', '인구_세대수', '인구_외국인', '인구_인구밀도', '인구_평균연령', '인구_한국인', '인구_고령인구비율', '인구_외국인비율', '인구_한국인비율']

연도별 인구통계 매칭률
                전체     매칭  매칭률(%)
_check_year                      
2016         14382  13301   92.48
2017         11491  10818   94.14
2018          9259   8732   94.31
2019          7568   6611   87.35
2020          7523   6761   89.87
2021          5496   5247   95.47
2022          5959   5764   96.73
2023          5595   5488   98.09
2024          5562   5256   94.50
2025          4901      0    0.00

 

In [8]:
df.to_csv(r'C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\business_crisis_cohort_v3.csv', index=False)

In [9]:
df.isnull().mean()

관리번호         0.000000
구            0.000000
읍면동          0.000000
업종           0.000000
기준분기         0.000000
               ...   
인구_평균연령      0.548124
인구_한국인       0.125527
인구_고령인구비율    0.125527
인구_외국인비율     0.125527
인구_한국인비율     0.125527
Length: 113, dtype: float64

In [10]:
df2 = pd.read_csv(r'C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\IMBANK파생변수.csv')

C:\Users\DC\AppData\Local\Temp\ipykernel_18128\2432294928.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(r'C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\IMBANK파생변수.csv')


In [11]:
df2.columns

Index(['Unnamed: 0', '관리번호', '사업장명', '기준연도', '기준일자', '업종', '도로명주소', '지번주소',
       '좌표정보(X)', '좌표정보(Y)', '경도', '위도', '시도', '구', '법정동', '법정동코드', '행정동',
       '행정동코드', '인허가일자', '인허가연도', '인허가월', '인허가분기', '인허가계절', '폐업일자', '폐업연도',
       '폐업월', '폐업분기', '폐업계절', '폐업상반기여부', '폐업연말여부', '폐업까지_영업일수', '폐업까지_영업개월수',
       '폐업까지_영업연수', '단기폐업_1년이내', '단기폐업_3년이내', '단기폐업_5년이내', '장기영업후폐업_10년이상',
       '해당연도_존재여부', '연말_영업여부', '해당연도_신규여부', '해당연도_폐업여부', '최근1년_신규여부',
       '최근3년_신규여부', '최근1년_폐업여부', '기준연도말_폐업상태', '영업일수', '영업개월수', '영업연수',
       '구_전체사업장수', '구_동일업종수', '구_동일업종비율', '동_전체사업장수', '동_동일업종수', '동_동일업종비율',
       '동_해당연도_동일업종신규수', '동_해당연도_동일업종폐업수', '동_해당연도_동일업종순증감', '동_동일업종_신규진입률',
       '동_동일업종_폐업률', '동_동일업종_평균업력', '지역대비_업력비율', '폐업여부_2015', '폐업여부_2016',
       '폐업여부_2017', '폐업여부_2018', '폐업여부_2019', '폐업여부_2020', '폐업여부_2021',
       '폐업여부_2022', '폐업여부_2023', '폐업여부_2024', '폐업여부_2025', '폐업여부_2026'],
      dtype='object')

In [12]:
df2 = df2[['관리번호','위도','경도']]

In [13]:
df = df.merge(df2, how='left', left_on='관리번호', right_on='관리번호')

In [14]:
df5 = pd.read_csv(r'C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\processed\variable\지하철\대구교통공사_도시철도 3호선 지상 엘리베이터 위치정보_20251114.csv', encoding='cp949')

In [15]:
df5

,호선,역명,EV번호,위도,경도
0,3,칠곡경대병원,1,35.958331,128.559762
1,3,칠곡경대병원,2,35.958310,128.560000
2,3,학정,1,35.951741,128.559103
3,3,학정,2,35.951729,128.559335
4,3,팔거,1,35.944243,128.558158
5,3,팔거,2,35.944186,128.558555
6,3,동천,1,35.937462,128.556578
7,3,칠곡운암,1,35.931617,128.554719
8,3,구암,1,35.925493,128.550303
9,3,태전,1,35.919465,128.547285


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


# ============================================================
# 1. 설정
# ============================================================

SUBWAY_DIR = Path(
    r"C:\Users\DC\2026\IMBANK\iM_Blockchain_AI"
    r"\data\processed\variable\지하철"
)

# 기준 데이터프레임:
# df
#
# 사업장 좌표:
# X = 경도
# Y = 위도


# ============================================================
# 2. 1~3호선 CSV 전체 읽기
# ============================================================

files = sorted(
    SUBWAY_DIR.glob("*.csv")
)

print("발견된 CSV 파일")

for file in files:
    print(" -", file.name)

if len(files) == 0:
    raise FileNotFoundError(
        f"CSV 파일을 찾을 수 없습니다: {SUBWAY_DIR}"
    )


station_list = []

for file in files:

    try:
        temp = pd.read_csv(
            file,
            encoding="utf-8-sig"
        )

    except UnicodeDecodeError:

        temp = pd.read_csv(
            file,
            encoding="cp949"
        )

    # 컬럼명 정리
    temp.columns = (
        temp.columns
        .astype(str)
        .str.strip()
    )

    # 필요한 컬럼 확인
    required_cols = {
        "호선",
        "역명",
        "EV번호",
        "위도",
        "경도"
    }

    missing = (
        required_cols
        - set(temp.columns)
    )

    if missing:
        print(
            f"[건너뜀] {file.name}: "
            f"없는 컬럼 = {missing}"
        )
        continue

    # 어느 파일에서 왔는지 기록
    temp["_source_file"] = file.name

    station_list.append(temp)


if len(station_list) == 0:
    raise RuntimeError(
        "정상적으로 읽은 지하철 CSV가 없습니다."
    )


station_raw = pd.concat(
    station_list,
    ignore_index=True
)


print()
print(
    "전체 EV 데이터:",
    station_raw.shape
)


# ============================================================
# 3. 역명 정규화
# ============================================================
#
# 예:
#
# 반월당
# 반월당1
# 반월당2
#
# → 반월당
#
# 동일 역이 여러 호선 파일에 있어도 같은 역으로 처리
# ============================================================

station_raw["역명_정규화"] = (
    station_raw["역명"]
    .astype(str)
    .str.strip()
    .str.replace(
        r"[1-3]$",
        "",
        regex=True
    )
)


# ============================================================
# 4. EV 좌표 정리
# ============================================================

station_raw["위도"] = pd.to_numeric(
    station_raw["위도"],
    errors="coerce"
)

station_raw["경도"] = pd.to_numeric(
    station_raw["경도"],
    errors="coerce"
)


# 좌표가 없는 EV 제거
station_ev = (
    station_raw
    .dropna(
        subset=[
            "위도",
            "경도"
        ]
    )
    .copy()
)


print(
    "좌표가 존재하는 EV:",
    len(station_ev)
)

print(
    "고유 역:",
    station_ev["역명_정규화"].nunique()
)


# ============================================================
# 5. 동일한 EV 좌표가 중복된 경우 제거
# ============================================================
#
# 같은 역 + 같은 위도 + 같은 경도
# 가 여러 번 들어있다면 실제 계산에서는 한 번만 사용
# ============================================================

station_ev = (
    station_ev
    .drop_duplicates(
        subset=[
            "역명_정규화",
            "위도",
            "경도"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. df 사업장 좌표 정리
# ============================================================
#
# X = 경도
# Y = 위도
# ============================================================

df["경도"] = pd.to_numeric(
    df["경도"],
    errors="coerce"
)

df["위도"] = pd.to_numeric(
    df["위도"],
    errors="coerce"
)


# ============================================================
# 7. 좌표 유효성 검사
# ============================================================
#
# 한국 좌표라면 대략:
#
# 위도 30~40
# 경도 120~135
#
# 범위를 넉넉하게 잡음
# ============================================================

valid_business = (
    df["경도"].between(
        120,
        135
    )
    &
    df["위도"].between(
        30,
        40
    )
)


valid_ev = (
    station_ev["경도"].between(
        120,
        135
    )
    &
    station_ev["위도"].between(
        30,
        40
    )
)

station_ev = (
    station_ev.loc[
        valid_ev
    ]
    .reset_index(drop=True)
)


print()
print(
    "좌표가 정상인 사업장:",
    valid_business.sum(),
    "/",
    len(df)
)

print(
    "좌표가 정상인 EV:",
    len(station_ev)
)


# ============================================================
# 8. Haversine 거리 계산
# ============================================================

def haversine_matrix(
    business_lat,
    business_lon,
    ev_lat,
    ev_lon
):

    """
    사업장 × EV 거리행렬 생성

    반환 단위:
        meter
    """

    R = 6_371_000

    business_lat = np.radians(
        np.asarray(
            business_lat,
            dtype=float
        )
    )[:, None]

    business_lon = np.radians(
        np.asarray(
            business_lon,
            dtype=float
        )
    )[:, None]

    ev_lat = np.radians(
        np.asarray(
            ev_lat,
            dtype=float
        )
    )[None, :]

    ev_lon = np.radians(
        np.asarray(
            ev_lon,
            dtype=float
        )
    )[None, :]


    dlat = (
        ev_lat
        - business_lat
    )

    dlon = (
        ev_lon
        - business_lon
    )


    a = (
        np.sin(
            dlat / 2
        ) ** 2
        +
        np.cos(
            business_lat
        )
        *
        np.cos(
            ev_lat
        )
        *
        np.sin(
            dlon / 2
        ) ** 2
    )


    c = (
        2
        *
        np.arctan2(
            np.sqrt(a),
            np.sqrt(1 - a)
        )
    )


    return R * c


# ============================================================
# 9. 유효한 사업장만 추출
# ============================================================

business_valid = (
    df.loc[
        valid_business,
        [
            "경도",
            "위도"
        ]
    ]
    .copy()
)


# ============================================================
# 10. 사업장 × 모든 EV 거리 계산
# ============================================================

distance_ev = haversine_matrix(

    business_valid["위도"].values,
    business_valid["경도"].values,

    station_ev["위도"].values,
    station_ev["경도"].values

)


print()
print(
    "사업장 × EV 거리 행렬:",
    distance_ev.shape
)


# ============================================================
# 11. EV → 고유 역 단위 거리로 변환
# ============================================================
#
# 핵심:
#
# 반월당역
#   EV1 = 80m
#   EV2 = 120m
#   EV3 = 210m
#
# → 해당 사업장과 반월당역 거리 = 80m
#
# 사업장마다 선택되는 EV가 달라도 됨
# ============================================================

station_names = (
    station_ev["역명_정규화"]
    .drop_duplicates()
    .tolist()
)


# 사업장 × 고유 역
station_distance = np.full(
    (
        len(business_valid),
        len(station_names)
    ),
    np.nan
)


for station_idx, station_name in enumerate(
    station_names
):

    # 해당 역의 모든 EV 위치
    ev_indices = np.where(
        station_ev[
            "역명_정규화"
        ].values
        ==
        station_name
    )[0]

    # 해당 사업장 → 해당 역의
    # 모든 EV 거리
    temp_distance = (
        distance_ev[
            :,
            ev_indices
        ]
    )

    # 가장 가까운 EV 선택
    station_distance[
        :,
        station_idx
    ] = np.min(
        temp_distance,
        axis=1
    )


print(
    "사업장 × 고유역 거리 행렬:",
    station_distance.shape
)


# ============================================================
# 12. 가장 가까운 역 찾기
# ============================================================

nearest_station_idx = np.argmin(
    station_distance,
    axis=1
)

nearest_station_distance = np.min(
    station_distance,
    axis=1
)

nearest_station_name = np.array(
    station_names
)[
    nearest_station_idx
]


# ============================================================
# 13. 접근성 변수 초기화
# ============================================================

df["지하철_최근접역"] = pd.NA

df["지하철_최근접거리_m"] = np.nan

df["지하철_100m_역수"] = pd.NA
df["지하철_300m_역수"] = pd.NA
df["지하철_500m_역수"] = pd.NA


# ============================================================
# 14. 최근접 역
# ============================================================

df.loc[
    valid_business,
    "지하철_최근접역"
] = nearest_station_name


# ============================================================
# 15. 최근접 역까지 거리
# ============================================================

df.loc[
    valid_business,
    "지하철_최근접거리_m"
] = nearest_station_distance


# ============================================================
# 16. 100m 이내 고유 역 개수
# ============================================================
#
# EV 개수가 아니라 "역 개수"
# ============================================================

df.loc[
    valid_business,
    "지하철_100m_역수"
] = (
    station_distance
    <= 100
).sum(axis=1)


# ============================================================
# 17. 300m 이내 고유 역 개수
# ============================================================

df.loc[
    valid_business,
    "지하철_300m_역수"
] = (
    station_distance
    <= 300
).sum(axis=1)


# ============================================================
# 18. 500m 이내 고유 역 개수
# ============================================================

df.loc[
    valid_business,
    "지하철_500m_역수"
] = (
    station_distance
    <= 500
).sum(axis=1)


# ============================================================
# 19. 역 개수 → nullable integer
# ============================================================

for col in [
    "지하철_100m_역수",
    "지하철_300m_역수",
    "지하철_500m_역수"
]:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).astype("Int64")


# ============================================================
# 20. 거리 반올림
# ============================================================

df["지하철_최근접거리_m"] = (
    df["지하철_최근접거리_m"]
    .round(1)
)


# ============================================================
# 21. 추가 변수 확인
# ============================================================

SUBWAY_COLS = [

    "지하철_최근접역",

    "지하철_최근접거리_m",

    "지하철_100m_역수",
    "지하철_300m_역수",
    "지하철_500m_역수"

]


print()
print("=" * 70)
print("지하철 접근성 변수")
print("=" * 70)

print(
    df[
        [
            "경도",
            "위도"
        ]
        +
        SUBWAY_COLS
    ].head(20)
)


# ============================================================
# 22. 기초통계
# ============================================================

print()
print("=" * 70)
print("기초통계")
print("=" * 70)

print(
    df[
        [
            "지하철_최근접거리_m",
            "지하철_100m_역수",
            "지하철_300m_역수",
            "지하철_500m_역수"
        ]
    ].describe()
)


# ============================================================
# 23. 반경별 사업장 비율 확인
# ============================================================

print()
print("=" * 70)
print("지하철 접근 가능 사업장 비율")
print("=" * 70)

for radius in [
    100,
    300,
    500
]:

    col = f"지하철_{radius}m_역수"

    accessible = (
        df[col] > 0
    )

    print(
        f"{radius}m 이내 역 존재: "
        f"{accessible.mean() * 100:.2f}%"
    )

발견된 CSV 파일
 - 대구교통공사_도시철도 1호선 지상 엘리베이터 위치정보_20251114.csv
 - 대구교통공사_도시철도 2호선 지상 엘리베이터 위치정보_20251114.csv
 - 대구교통공사_도시철도 3호선 지상 엘리베이터 위치정보_20251114.csv

전체 EV 데이터: (152, 6)
좌표가 존재하는 EV: 152
고유 역: 87

좌표가 정상인 사업장: 591355 / 596144
좌표가 정상인 EV: 152

사업장 × EV 거리 행렬: (591355, 152)
사업장 × 고유역 거리 행렬: (591355, 87)

지하철 접근성 변수
            경도         위도 지하철_최근접역  지하철_최근접거리_m  지하철_100m_역수  지하철_300m_역수  \
0   128.616536  35.865351     대구은행        621.4            0            0   
1   128.616536  35.865351     대구은행        621.4            0            0   
2   128.616536  35.865351     대구은행        621.4            0            0   
3   128.616536  35.865351     대구은행        621.4            0            0   
4   128.616536  35.865351     대구은행        621.4            0            0   
5   128.616536  35.865351     대구은행        621.4            0            0   
6   128.616536  35.865351     대구은행        621.4         

In [18]:
df.to_csv(r'C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\business_crisis_cohort_v4.csv', index=False)

In [19]:
df

,관리번호,구,읍면동,업종,기준분기,t0,업력_일수,매출액,이용건수,sales_yoy,...,인구_고령인구비율,인구_외국인비율,인구_한국인비율,위도,경도,지하철_최근접역,지하철_최근접거리_m,지하철_100m_역수,지하철_300m_역수,지하철_500m_역수
0,3460000-104-2010-00068,수성구,수성4가동,식품_휴게음식점,2016Q1,2016-03-31,2095,142855816.0,17613,-0.294233,...,0.146061,0.003668,0.996332,35.865351,128.616536,대구은행,621.4,0,0,0
1,3460000-104-2010-00068,수성구,수성4가동,식품_휴게음식점,2016Q1,2016-03-31,2095,142855816.0,17613,-0.294233,...,0.146061,0.003668,0.996332,35.865351,128.616536,대구은행,621.4,0,0,0
2,3460000-104-2010-00068,수성구,수성4가동,식품_휴게음식점,2016Q1,2016-03-31,2095,142855816.0,17613,-0.294233,...,0.146061,0.003668,0.996332,35.865351,128.616536,대구은행,621.4,0,0,0
3,3460000-104-2010-00068,수성구,수성4가동,식품_휴게음식점,2016Q1,2016-03-31,2095,142855816.0,17613,-0.294233,...,0.146061,0.003668,0.996332,35.865351,128.616536,대구은행,621.4,0,0,0
4,3460000-104-2010-00068,수성구,수성4가동,식품_휴게음식점,2016Q1,2016-03-31,2095,142855816.0,17613,-0.294233,...,0.146061,0.003668,0.996332,35.865351,128.616536,대구은행,621.4,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
596139,3480000-101-2022-00177,달성군,다사읍,식품_일반음식점,2025Q4,2025-12-31,1223,600237283.0,20546,-0.442391,...,NaN,NaN,NaN,35.859386,128.465164,대실,236.4,0,1,1
596140,3480000-101-2022-00177,달성군,다사읍,식품_일반음식점,2025Q4,2025-12-31,1223,600237283.0,20546,-0.442391,...,NaN,NaN,NaN,35.859386,128.465164,대실,236.4,0,1,1
596141,3480000-101-2022-00177,달성군,다사읍,식품_일반음식점,2025Q4,2025-12-31,1223,600237283.0,20546,-0.442391,...,NaN,NaN,NaN,35.859386,128.465164,대실,236.4,0,1,1
596142,CDFH3301082025000004,달서구,도원동,생활_당구장업,2025Q4,2025-12-31,204,316141800.0,9652,-0.205709,...,NaN,NaN,NaN,35.808939,128.536437,월배,950.6,0,0,0


In [20]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


# ============================================================
# 1. CONFIG
# ============================================================

RIDERSHIP_DIR = Path(
    r"C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\processed\variable\지하철승하차"
)

# df에는 이미 아래 컬럼이 있다고 가정
#
# t0
# 지하철_최근접역
# 지하철_최근접거리_m
# 지하철_100m_역수
# 지하철_300m_역수
# 지하철_500m_역수


# ============================================================
# 2. 함수
# ============================================================

def normalize_station_name(x):
    """
    동일 역의 끝자리 번호 제거

    예:
    반월당1 -> 반월당
    반월당2 -> 반월당
    반월당3 -> 반월당
    """

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # 역명 뒤 1~3 제거
    x = re.sub(r"[1-3]$", "", x)

    return x


def read_file(path):
    """
    CSV / Excel 자동 읽기
    """

    suffix = path.suffix.lower()

    if suffix == ".csv":

        for enc in [
            "utf-8-sig",
            "cp949",
            "euc-kr"
        ]:

            try:
                return pd.read_csv(
                    path,
                    encoding=enc
                )

            except UnicodeDecodeError:
                continue

        raise ValueError(
            f"CSV 인코딩 확인 필요: {path}"
        )

    elif suffix in [
        ".xlsx",
        ".xls"
    ]:

        return pd.read_excel(path)

    else:

        raise ValueError(
            f"지원하지 않는 파일 형식: {path}"
        )


# ============================================================
# 3. 승하차 파일 찾기
# ============================================================

files = (
    list(RIDERSHIP_DIR.glob("*.csv"))
    + list(RIDERSHIP_DIR.glob("*.xlsx"))
    + list(RIDERSHIP_DIR.glob("*.xls"))
)

files = sorted(files)

print("=" * 60)
print("승하차 파일")
print("=" * 60)

for f in files:
    print(f.name)

print("\n파일 수:", len(files))


if len(files) == 0:
    raise FileNotFoundError(
        f"승하차 파일이 없습니다: {RIDERSHIP_DIR}"
    )


# ============================================================
# 4. 전체 파일 읽기
# ============================================================

frames = []

for file in files:

    temp = read_file(file)

    # 컬럼명 정리
    temp.columns = (
        temp.columns
        .astype(str)
        .str.strip()
    )

    # --------------------------------------------------------
    # 파일명에서 연도 추출
    #
    # 예:
    # 지하철승하차_2019.csv
    # 2020년승하차.xlsx
    # -> 2019 / 2020
    # --------------------------------------------------------

    year_match = re.search(
        r"(20\d{2})",
        file.name
    )

    if year_match is None:

        raise ValueError(
            f"\n파일명에서 연도를 찾을 수 없습니다.\n"
            f"파일명: {file.name}"
        )

    year = int(
        year_match.group(1)
    )

    temp["_연도"] = year
    temp["_source_file"] = file.name

    frames.append(temp)


ridership_raw = pd.concat(
    frames,
    ignore_index=True
)


print("\n" + "=" * 60)
print("원본 데이터")
print("=" * 60)

print(
    "원본 shape:",
    ridership_raw.shape
)

print(
    "\n컬럼:",
    ridership_raw.columns.tolist()
)

print(
    "\n연도별 행 수:"
)

print(
    ridership_raw["_연도"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 5. 필수 컬럼 확인
# ============================================================

required_cols = [
    "월",
    "일",
    "역명",
    "승하"
]

missing_cols = [
    col
    for col in required_cols
    if col not in ridership_raw.columns
]

if missing_cols:

    raise ValueError(
        f"필수 컬럼이 없습니다: {missing_cols}"
    )


# ============================================================
# 6. 기본 전처리
# ============================================================

ridership = ridership_raw.copy()


# ------------------------------------------------------------
# 역명 정규화
# ------------------------------------------------------------

ridership["역명_정규화"] = (
    ridership["역명"]
    .apply(normalize_station_name)
)


# ------------------------------------------------------------
# 월 / 일 숫자 변환
# ------------------------------------------------------------

ridership["월"] = pd.to_numeric(
    ridership["월"],
    errors="coerce"
)

ridership["일"] = pd.to_numeric(
    ridership["일"],
    errors="coerce"
)


# ------------------------------------------------------------
# 날짜 생성
#
# 파일명 연도 + 월 + 일
# ------------------------------------------------------------

ridership["날짜"] = pd.to_datetime(
    {
        "year": ridership["_연도"],
        "month": ridership["월"],
        "day": ridership["일"]
    },
    errors="coerce"
)


print("\n" + "=" * 60)
print("날짜 확인")
print("=" * 60)

print(
    "날짜 변환 실패:",
    ridership["날짜"].isna().sum()
)

print(
    ridership[
        [
            "_source_file",
            "_연도",
            "월",
            "일",
            "날짜"
        ]
    ].head()
)


# ============================================================
# 7. 시간대 컬럼 자동 탐색
# ============================================================

# 예:
# 05시-06시
# 06시-07시
# ...
# 23시-24시

hour_cols = [
    c
    for c in ridership.columns
    if re.match(
        r"^\d{2}시-\d{2}시$",
        str(c)
    )
]


print("\n" + "=" * 60)
print("시간대 컬럼")
print("=" * 60)

print(
    "시간대 컬럼 수:",
    len(hour_cols)
)

print(hour_cols)


if len(hour_cols) == 0:

    raise ValueError(
        "시간대 컬럼을 찾지 못했습니다."
    )


# ============================================================
# 8. 숫자형 변환
# ============================================================

numeric_cols = hour_cols.copy()

if "일계" in ridership.columns:
    numeric_cols.append("일계")


for col in numeric_cols:

    ridership[col] = (
        ridership[col]
        .astype(str)
        .str.replace(
            ",",
            "",
            regex=False
        )
        .str.strip()
        .replace(
            {
                "": np.nan,
                "-": np.nan,
                "nan": np.nan,
                "None": np.nan
            }
        )
    )

    ridership[col] = pd.to_numeric(
        ridership[col],
        errors="coerce"
    )


# ============================================================
# 9. 시간대 합 계산
# ============================================================

ridership["_시간대합"] = (
    ridership[hour_cols]
    .sum(
        axis=1,
        min_count=1
    )
)


# ============================================================
# 10. 일계와 시간대 합 검증
# ============================================================

if "일계" in ridership.columns:

    ridership["_일계차이"] = (
        ridership["일계"]
        - ridership["_시간대합"]
    )

    diff_count = (
        ridership["_일계차이"]
        .abs()
        .gt(0.01)
        .sum()
    )

    print("\n" + "=" * 60)
    print("일계 검증")
    print("=" * 60)

    print(
        "시간대 합 != 일계 행 수:",
        diff_count
    )

    print(
        "전체 행:",
        len(ridership)
    )


# ============================================================
# 11. 승 / 하 값 확인
# ============================================================

ridership["승하"] = (
    ridership["승하"]
    .astype(str)
    .str.strip()
)


print("\n" + "=" * 60)
print("승하 값")
print("=" * 60)

print(
    ridership["승하"]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# 12. 승차 / 하차 변수 생성
# ============================================================

# 모든 계산의 기준을 시간대 합으로 통일
ridership["_총승하차"] = (
    ridership["_시간대합"]
)


# ------------------------------------------------------------
# 승차
# ------------------------------------------------------------

ridership["_승차"] = np.where(
    ridership["승하"]
    .str.contains(
        "승",
        na=False
    ),
    ridership["_총승하차"],
    0
)


# ------------------------------------------------------------
# 하차
# ------------------------------------------------------------

ridership["_하차"] = np.where(
    ridership["승하"]
    .str.contains(
        "하",
        na=False
    ),
    ridership["_총승하차"],
    0
)


# ============================================================
# 13. 시간대 구분
# ============================================================
#
# 새벽/이른아침 : 05 ~ 07
# 출근          : 07 ~ 10
# 주간          : 10 ~ 17
# 퇴근          : 17 ~ 20
# 야간          : 20 ~ 24
#
# 전체 시간대를 겹치지 않게 분할
# ============================================================


def get_hour_start(col):

    match = re.match(
        r"^(\d{2})시-",
        col
    )

    return int(
        match.group(1)
    )


early_cols = [
    c
    for c in hour_cols
    if 5 <= get_hour_start(c) < 7
]


morning_cols = [
    c
    for c in hour_cols
    if 7 <= get_hour_start(c) < 10
]


day_cols = [
    c
    for c in hour_cols
    if 10 <= get_hour_start(c) < 17
]


evening_cols = [
    c
    for c in hour_cols
    if 17 <= get_hour_start(c) < 20
]


night_cols = [
    c
    for c in hour_cols
    if 20 <= get_hour_start(c) < 24
]


print("\n" + "=" * 60)
print("시간대 구분")
print("=" * 60)

print("이른아침:", early_cols)
print("출근:", morning_cols)
print("주간:", day_cols)
print("퇴근:", evening_cols)
print("야간:", night_cols)


# ============================================================
# 14. 시간대별 승하차 합
# ============================================================

ridership["_이른아침"] = (
    ridership[early_cols]
    .sum(
        axis=1,
        min_count=1
    )
)

ridership["_출근"] = (
    ridership[morning_cols]
    .sum(
        axis=1,
        min_count=1
    )
)

ridership["_주간"] = (
    ridership[day_cols]
    .sum(
        axis=1,
        min_count=1
    )
)

ridership["_퇴근"] = (
    ridership[evening_cols]
    .sum(
        axis=1,
        min_count=1
    )
)

ridership["_야간"] = (
    ridership[night_cols]
    .sum(
        axis=1,
        min_count=1
    )
)


# ============================================================
# 15. 같은 역 / 같은 날짜 통합
# ============================================================
#
# 반월당1
# 반월당2
# 반월당3
#
# -> 반월당
#
# 같은 물리적 역의 승하차량은 합산
# ============================================================

daily = (
    ridership
    .groupby(
        [
            "날짜",
            "역명_정규화"
        ],
        as_index=False
    )
    .agg(
        총승하차=(
            "_총승하차",
            "sum"
        ),

        승차=(
            "_승차",
            "sum"
        ),

        하차=(
            "_하차",
            "sum"
        ),

        이른아침=(
            "_이른아침",
            "sum"
        ),

        출근시간=(
            "_출근",
            "sum"
        ),

        주간시간=(
            "_주간",
            "sum"
        ),

        퇴근시간=(
            "_퇴근",
            "sum"
        ),

        야간시간=(
            "_야간",
            "sum"
        )
    )
)


print("\n" + "=" * 60)
print("일별 역 단위 집계")
print("=" * 60)

print(
    "shape:",
    daily.shape
)

print(
    daily.head()
)


# ============================================================
# 16. YYYYMM 생성
# ============================================================

daily["YYYYMM"] = (
    daily["날짜"]
    .dt.strftime("%Y%m")
)


# ============================================================
# 17. 역별 월 집계
# ============================================================

monthly = (
    daily
    .groupby(
        [
            "YYYYMM",
            "역명_정규화"
        ],
        as_index=False
    )
    .agg(

        관측일수=(
            "날짜",
            "nunique"
        ),

        월승하차=(
            "총승하차",
            "sum"
        ),

        월승차=(
            "승차",
            "sum"
        ),

        월하차=(
            "하차",
            "sum"
        ),

        월이른아침=(
            "이른아침",
            "sum"
        ),

        월출근=(
            "출근시간",
            "sum"
        ),

        월주간=(
            "주간시간",
            "sum"
        ),

        월퇴근=(
            "퇴근시간",
            "sum"
        ),

        월야간=(
            "야간시간",
            "sum"
        )
    )
)


# ============================================================
# 18. 일평균 계산
# ============================================================
#
# 월 총합 / 실제 관측일수
#
# 2월, 3월 등 월 길이 차이 제거
# 데이터 누락이 있어도 실제 관측일수 기준
# ============================================================

monthly[
    "지하철_일평균승하차"
] = (
    monthly["월승하차"]
    / monthly["관측일수"]
)


monthly[
    "지하철_일평균승차"
] = (
    monthly["월승차"]
    / monthly["관측일수"]
)


monthly[
    "지하철_일평균하차"
] = (
    monthly["월하차"]
    / monthly["관측일수"]
)


monthly[
    "지하철_이른아침_일평균"
] = (
    monthly["월이른아침"]
    / monthly["관측일수"]
)


monthly[
    "지하철_출근시간_일평균"
] = (
    monthly["월출근"]
    / monthly["관측일수"]
)


monthly[
    "지하철_주간시간_일평균"
] = (
    monthly["월주간"]
    / monthly["관측일수"]
)


monthly[
    "지하철_퇴근시간_일평균"
] = (
    monthly["월퇴근"]
    / monthly["관측일수"]
)


monthly[
    "지하철_야간시간_일평균"
] = (
    monthly["월야간"]
    / monthly["관측일수"]
)


# ============================================================
# 19. 시간대 비율
# ============================================================
#
# 예:
# 출근시간 비율 =
# 출근시간 승하차 / 전체 승하차
#
# 역의 절대 규모가 아니라
# 시간대별 이용 특성을 나타냄
# ============================================================

denom = (
    monthly["월승하차"]
    .replace(
        0,
        np.nan
    )
)


monthly[
    "지하철_이른아침_비율"
] = (
    monthly["월이른아침"]
    / denom
)


monthly[
    "지하철_출근시간_비율"
] = (
    monthly["월출근"]
    / denom
)


monthly[
    "지하철_주간시간_비율"
] = (
    monthly["월주간"]
    / denom
)


monthly[
    "지하철_퇴근시간_비율"
] = (
    monthly["월퇴근"]
    / denom
)


monthly[
    "지하철_야간시간_비율"
] = (
    monthly["월야간"]
    / denom
)


# ============================================================
# 20. 승차 / 하차 비율
# ============================================================

monthly[
    "지하철_승차비율"
] = (
    monthly["월승차"]
    / denom
)


monthly[
    "지하철_하차비율"
] = (
    monthly["월하차"]
    / denom
)


# ============================================================
# 21. 월별 데이터 검증
# ============================================================

print("\n" + "=" * 60)
print("월별 데이터 검증")
print("=" * 60)


print(
    "월 범위:",
    monthly["YYYYMM"].min(),
    "~",
    monthly["YYYYMM"].max()
)


print("\n관측일수:")
print(
    monthly["관측일수"]
    .describe()
)


print("\n역 수:")
print(
    monthly["역명_정규화"]
    .nunique()
)


# ============================================================
# 22. 시간대 비율 합 검증
# ============================================================

monthly["_시간대비율합"] = (
    monthly[
        [
            "지하철_이른아침_비율",
            "지하철_출근시간_비율",
            "지하철_주간시간_비율",
            "지하철_퇴근시간_비율",
            "지하철_야간시간_비율"
        ]
    ]
    .sum(
        axis=1,
        min_count=1
    )
)


print(
    "\n시간대 비율 합:"
)

print(
    monthly["_시간대비율합"]
    .describe()
)


# ============================================================
# 23. 최종 역-월 데이터
# ============================================================

monthly_final = monthly[
    [
        "YYYYMM",
        "역명_정규화",

        "관측일수",

        "지하철_일평균승하차",
        "지하철_일평균승차",
        "지하철_일평균하차",

        "지하철_이른아침_일평균",
        "지하철_출근시간_일평균",
        "지하철_주간시간_일평균",
        "지하철_퇴근시간_일평균",
        "지하철_야간시간_일평균",

        "지하철_이른아침_비율",
        "지하철_출근시간_비율",
        "지하철_주간시간_비율",
        "지하철_퇴근시간_비율",
        "지하철_야간시간_비율",

        "지하철_승차비율",
        "지하철_하차비율"
    ]
].copy()


# ============================================================
# 24. 중복 검증
# ============================================================

dup = (
    monthly_final
    .duplicated(
        subset=[
            "YYYYMM",
            "역명_정규화"
        ]
    )
    .sum()
)


print(
    "\n역-월 중복:",
    dup
)


if dup > 0:

    raise ValueError(
        "역-월 단위 중복이 존재합니다."
    )


# ============================================================
# 25. df 날짜 생성
# ============================================================

df["t0"] = pd.to_datetime(
    df["t0"],
    errors="coerce"
)


df["_YYYYMM"] = (
    df["t0"]
    .dt.strftime("%Y%m")
)


# ============================================================
# 26. df 최근접역 정규화
# ============================================================

df["_최근접역_정규화"] = (
    df["지하철_최근접역"]
    .apply(
        normalize_station_name
    )
)


# ============================================================
# 27. 역명 매칭 확인
# ============================================================

df_station_set = set(
    df["_최근접역_정규화"]
    .dropna()
    .unique()
)


ridership_station_set = set(
    monthly_final["역명_정규화"]
    .dropna()
    .unique()
)


matched_station = (
    df_station_set
    & ridership_station_set
)


unmatched_station = (
    df_station_set
    - ridership_station_set
)


print("\n" + "=" * 60)
print("역명 매칭")
print("=" * 60)


print(
    "df 최근접역 수:",
    len(df_station_set)
)

print(
    "승하차 역 수:",
    len(ridership_station_set)
)

print(
    "매칭 역 수:",
    len(matched_station)
)


print(
    "\n매칭 안 된 역:"
)

print(
    sorted(unmatched_station)
)


# ============================================================
# 28. df와 월별 승하차 데이터 Merge
# ============================================================

before_rows = len(df)


df = df.merge(
    monthly_final,

    how="left",

    left_on=[
        "_YYYYMM",
        "_최근접역_정규화"
    ],

    right_on=[
        "YYYYMM",
        "역명_정규화"
    ],

    validate="m:1"
)


# ============================================================
# 29. 행 수 보존 검증
# ============================================================

if len(df) != before_rows:

    raise ValueError(
        f"""
        Merge 후 행 수가 변경되었습니다.

        이전: {before_rows}
        이후: {len(df)}
        """
    )


# ============================================================
# 30. 최근접역 변수로 이름 변경
# ============================================================

rename_map = {

    "관측일수":
        "지하철_최근접역_관측일수",

    "지하철_일평균승하차":
        "지하철_최근접역_일평균승하차",

    "지하철_일평균승차":
        "지하철_최근접역_일평균승차",

    "지하철_일평균하차":
        "지하철_최근접역_일평균하차",

    "지하철_이른아침_일평균":
        "지하철_최근접역_이른아침_일평균",

    "지하철_출근시간_일평균":
        "지하철_최근접역_출근시간_일평균",

    "지하철_주간시간_일평균":
        "지하철_최근접역_주간시간_일평균",

    "지하철_퇴근시간_일평균":
        "지하철_최근접역_퇴근시간_일평균",

    "지하철_야간시간_일평균":
        "지하철_최근접역_야간시간_일평균",

    "지하철_이른아침_비율":
        "지하철_최근접역_이른아침_비율",

    "지하철_출근시간_비율":
        "지하철_최근접역_출근시간_비율",

    "지하철_주간시간_비율":
        "지하철_최근접역_주간시간_비율",

    "지하철_퇴근시간_비율":
        "지하철_최근접역_퇴근시간_비율",

    "지하철_야간시간_비율":
        "지하철_최근접역_야간시간_비율",

    "지하철_승차비율":
        "지하철_최근접역_승차비율",

    "지하철_하차비율":
        "지하철_최근접역_하차비율"
}


df = df.rename(
    columns=rename_map
)


# ============================================================
# 31. 임시 컬럼 제거
# ============================================================

df = df.drop(
    columns=[
        "_YYYYMM",
        "_최근접역_정규화",
        "YYYYMM",
        "역명_정규화"
    ],
    errors="ignore"
)


# ============================================================
# 32. 새로 추가된 변수
# ============================================================

new_cols = [

    "지하철_최근접역_관측일수",

    "지하철_최근접역_일평균승하차",
    "지하철_최근접역_일평균승차",
    "지하철_최근접역_일평균하차",

    "지하철_최근접역_이른아침_일평균",
    "지하철_최근접역_출근시간_일평균",
    "지하철_최근접역_주간시간_일평균",
    "지하철_최근접역_퇴근시간_일평균",
    "지하철_최근접역_야간시간_일평균",

    "지하철_최근접역_이른아침_비율",
    "지하철_최근접역_출근시간_비율",
    "지하철_최근접역_주간시간_비율",
    "지하철_최근접역_퇴근시간_비율",
    "지하철_최근접역_야간시간_비율",

    "지하철_최근접역_승차비율",
    "지하철_최근접역_하차비율"
]


# ============================================================
# 33. 최종 결과 확인
# ============================================================

print("\n" + "=" * 60)
print("최종 결과")
print("=" * 60)


print(
    "df shape:",
    df.shape
)


print("\n추가 변수:")

for col in new_cols:
    print(col)


# ============================================================
# 34. 결측률 확인
# ============================================================

print("\n" + "=" * 60)
print("승하차 변수 결측률")
print("=" * 60)


print(
    df[new_cols]
    .isna()
    .mean()
    .sort_values(
        ascending=False
    )
)


# ============================================================
# 35. 연도별 매칭률
# ============================================================

df["_연도_확인"] = (
    df["t0"]
    .dt.year
)


match_check = (
    df.groupby(
        "_연도_확인"
    )
    ["지하철_최근접역_일평균승하차"]
    .agg(
        전체="size",
        매칭="count"
    )
)


match_check["매칭률"] = (
    match_check["매칭"]
    / match_check["전체"]
)


print("\n" + "=" * 60)
print("연도별 승하차 데이터 매칭률")
print("=" * 60)

print(match_check)


df = df.drop(
    columns="_연도_확인"
)


# ============================================================
# 36. 샘플 확인
# ============================================================

show_cols = [
    "t0",
    "읍면동",
    "지하철_최근접역",
    "지하철_최근접거리_m",

    "지하철_최근접역_일평균승하차",
    "지하철_최근접역_일평균승차",
    "지하철_최근접역_일평균하차",

    "지하철_최근접역_출근시간_일평균",
    "지하철_최근접역_퇴근시간_일평균",

    "지하철_최근접역_출근시간_비율",
    "지하철_최근접역_퇴근시간_비율"
]


show_cols = [
    c
    for c in show_cols
    if c in df.columns
]


print("\n" + "=" * 60)
print("최종 샘플")
print("=" * 60)


print(
    df[show_cols]
    .head(20)
)


# ============================================================
# 37. 이상치 간단 확인
# ============================================================

print("\n" + "=" * 60)
print("일평균 승하차 기술통계")
print("=" * 60)


print(
    df[
        "지하철_최근접역_일평균승하차"
    ]
    .describe()
)


print("\n완료")

승하차 파일
2019.csv
2020.csv
2021.csv
2022.csv
2023.csv
2024.csv
2025.csv

파일 수: 7

원본 데이터
원본 shape: (473392, 28)

컬럼: ['월', '일', '역번호', '역명', '승하', '05시-06시', '06시-07시', '07시-08시', '08시-09시', '09시-10시', '10시-11시', '11시-12시', '12시-13시', '13시-14시', '14시-15시', '15시-16시', '16시-17시', '17시-18시', '18시-19시', '19시-20시', '20시-21시', '21시-22시', '22시-23시', '23시-24시', '일계', '_연도', '_source_file', '승하차']

연도별 행 수:
_연도
2019    66430
2020    66612
2021    66430
2022    66430
2023    72072
2024    66798
2025    68620
Name: count, dtype: int64

날짜 확인
날짜 변환 실패: 0
  _source_file   _연도  월  일         날짜
0     2019.csv  2019  1  1 2019-01-01
1     2019.csv  2019  1  1 2019-01-01
2     2019.csv  2019  1  1 2019-01-01
3     2019.csv  2019  1  1 2019-01-01
4     2019.csv  2019  1  1 2019-01-01

시간대 컬럼
시간대 컬럼 수: 19
['05시-06시', '06시-07시', '07시-08시', '08시-09시', '09시-10시', '10시-11시', '11시-12시', '12시-13시', '13시-14시', '14시-15시', '15시-16시', '16시-17시', '17시-18시', '18시-19시', '19시-20시', '20시-21시', '21시-22시', '22시-23시', '23시-

In [22]:
df.to_csv(r'C:\Users\DC\2026\IMBANK\iM_Blockchain_AI\data\business_crisis_cohort_v5.csv', index=False)